### Neural Probabilitic Language Model
This notebook provides an implementation of [Bengio et al (2003)](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf) in PyTorch and compares it with an n-gram model that was the prevailing statistical language model in 2003.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import trange
import nltk
nltk.download('brown')
from nltk.corpus import brown
from collections import Counter, defaultdict
from nltk.tokenize import word_tokenize
import random
import math
from RNNUtils import plot_errors_accuracy

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

### Train the N-Gram model

In [ ]:
corpus_sentences = brown.sents()

In [ ]:
corpus_sentences

In [ ]:
split_ratio = 0.8
split_index = int(len(corpus_sentences) * split_ratio)

In [ ]:
train_corpus_sentences = corpus_sentences[:split_index]
test_corpus_sentences = corpus_sentences[split_index:]

In [ ]:
tokenized_text = [word.lower() for sent in train_corpus_sentences for word in sent]
test_tokenized_text = [word.lower() for sent in test_corpus_sentences for word in sent]

In [ ]:
def preprocess_corpus(corpus):
    # Convert to lowercase and flatten the list of sentences
    return [word.lower() for sentence in corpus for word in sentence]

def create_ngrams(words, n=2):
    ngrams = list(zip(*[words[i:] for i in range(n)]))
    return ngrams

def train_ngram_model(corpus, n=2):
    # Preprocess and create ngrams
    words = preprocess_corpus(corpus)
    ngrams = create_ngrams(words, n)

    # Calculate frequencies
    ngram_freqs = Counter(ngrams)
    context_freqs = Counter([ngram[:-1] for ngram in ngrams])

    # Calculate probabilities
    model = defaultdict(lambda: defaultdict(float))
    for ngram, freq in ngram_freqs.items():
        context = ngram[:-1]
        word = ngram[-1]
        model[context][word] = freq / context_freqs[context]

    return model

In [ ]:
# Train the model
context_size = 3
n = context_size + 1
ngram_model = train_ngram_model(train_corpus_sentences, n)

In [ ]:
ngram_model

In [ ]:
def preprocess_sentence(sentence):
    # Convert to lowercase and flatten the list of sentences
    sentence = word_tokenize(sentence)
    return [word.lower() for word in sentence]

def predict_next_word(model, context, n=2):
    processed_context = preprocess_sentence(context)

    # Ensure the context is the right length for the model
    if len(processed_context) < n - 1:
        raise ValueError(f"Context must be at least {n - 1} words for an {n}-gram model.")

    # Use the last (n-1) words as the context for prediction
    ngram_context = tuple(processed_context[-(n - 1):])

    # Check if the context is in the model
    if ngram_context not in model:
        return None

    # Find the word with the highest probability
    possible_words = model[ngram_context]
    next_word = max(possible_words, key=possible_words.get)

    return next_word

In [ ]:
context = "that the city"
predicted_word = predict_next_word(ngram_model, context, n=n)
print(f"The predicted next word is: {predicted_word}")

In [ ]:
def calculate_perplexity_ngram(model, test_data, n):
    total_log_prob = 0
    total_word_count = 0
    for i in range(n - 1, len(test_data)):
        context = tuple(test_data[i - n + 1:i])
        word = test_data[i]
        word_probability = model[context].get(word, 0)  # Get the probability, 0 if not found
        total_log_prob += math.log(word_probability + 1e-12)  # Add a small value to avoid log(0)
        total_word_count += 1

    avg_log_prob = total_log_prob / total_word_count
    perplexity = math.exp(-avg_log_prob)
    return perplexity

In [ ]:
perplexity_ngram = calculate_perplexity_ngram(ngram_model, tokenized_text, n)
print(f'N-Gram Model Perplexity: {perplexity_ngram}')

In [ ]:
perplexity_ngram = calculate_perplexity_ngram(ngram_model, test_tokenized_text, n)
print(f'N-Gram Model Perplexity: {perplexity_ngram}')

### Train the Neural Network Model

In [ ]:
class BengioLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, context_size):
        super(BengioLanguageModel, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(context_size * embedding_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs)
        embeds = embeds.view(embeds.size(0), -1)
        out = torch.relu(self.linear1(embeds))
        return self.linear2(out)

In [ ]:
# Hyperparameters
embedding_dim = 64
hidden_dim = 64

In [ ]:
def build_vocab_and_index(data, min_freq, unk_token="<UNK>"):
    # Count the frequency of each token in the data
    counter = Counter(data)

    # Create vocabulary: map each frequent word to a unique index and 
    # infrequent words to the 'unknown' token index
    unk_index = 0  # Typically, the unknown token is indexed at 0
    vocab = {unk_token: unk_index}
    idx_to_word = {unk_index: unk_token}
    idx = 1
    
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = idx
            idx_to_word[idx] = word
            idx += 1
        else:
            # If a word is infrequent, map it to the unknown token's index
            vocab[word] = unk_index

    return vocab, idx_to_word

min_freq = 4
vocab, idx_to_word = build_vocab_and_index(tokenized_text, min_freq)

In [ ]:
vocab

In [ ]:
model = BengioLanguageModel(len(vocab), embedding_dim, hidden_dim, context_size).to(device)

In [ ]:
def create_context_target_pairs(data, context_size, vocab):
    for i in range(context_size, len(data)):
        context = [vocab.get(data[j], vocab['<UNK>']) for j in range(i - context_size, i)]
        target = vocab.get(data[i], vocab['<UNK>'])
        yield context, target

In [ ]:
class BengioDataset(Dataset):
    def __init__(self, corpus, context_size, vocab, device):
        self.data = [(torch.tensor(context, dtype=torch.long, device=device), 
                      torch.tensor(target, dtype=torch.long, device=device))
                     for context, target in create_context_target_pairs(corpus, context_size, vocab)]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [ ]:
batch_size = 512

train_dataset = BengioDataset(tokenized_text, context_size, vocab, device)
test_dataset = BengioDataset(test_tokenized_text, context_size, vocab, device)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

In [ ]:
def train_language_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_losses = []
    test_losses = []

    tqdm_epoch = trange(epochs, desc='Training')
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0
        correct_train = 0
        total_train = 0

        # Training loop
        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs, batch_y)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)
            _, predicted = torch.max(outputs, 1)
            correct_train += (predicted == batch_y).sum().item()
            total_train += batch_y.size(0)

        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        # Evaluation loop
        model.eval()
        test_loss = 0.0
        correct_test = 0
        total_test = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs, batch_y)
                test_loss += loss.item() * batch_X.size(0)
                _, predicted = torch.max(outputs, 1)
                correct_test += (predicted == batch_y).sum().item()
                total_test += batch_y.size(0)

        test_loss /= len(test_loader.dataset)
        test_losses.append(test_loss)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")

    return train_losses, test_losses


In [ ]:
lr = 1e-3
epochs = 20
optimizer = optim.Adam(model.parameters(), lr=lr)
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
errors_acc = train_language_model(model, train_loader, test_loader, 
                                            loss_fn, optimizer, epochs)

In [ ]:
def predict_next_word_nn(model, context, vocab, idx_to_word, context_size, device):
    # Ensure the model is in evaluation mode
    model.eval()

    # Convert context words to indices, handling unknown words
    context_indices = [vocab.get(word, vocab['<UNK>']) for word in context]

    # Ensure the context is of the right size for the model
    if len(context_indices) != context_size:
        raise ValueError(f"Context should be exactly {model.context_size} words.")

    # Convert to tensor and add batch dimension
    context_tensor = torch.tensor([context_indices], dtype=torch.long).to(device)

    # Get model output and convert to probability distribution
    with torch.no_grad():
        output = model(context_tensor)
        probabilities = torch.softmax(output, dim=1)

    # Get the most likely next word index
    predicted_index = torch.argmax(probabilities, dim=1).item()

    # Convert index to word
    predicted_word = idx_to_word[predicted_index]

    return predicted_word

In [ ]:
context = ["that", "the", "city"]

predicted_word = predict_next_word_nn(model, context, vocab, idx_to_word, context_size, device)
print("Predicted next word:", predicted_word)

In [ ]:
def calculate_perplexity(model, data_loader, device):
    model.eval()
    total_loss = 0.0
    total_words = 0

    with torch.no_grad():
        for batch_X, batch_y in data_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            outputs = model(batch_X)
            loss = F.cross_entropy(outputs, batch_y, reduction='sum')
            total_loss += loss.item()

            # Count the total number of words in the target
            total_words += batch_y.numel()

    average_loss = total_loss / total_words
    perplexity = math.exp(average_loss)

    return perplexity

In [ ]:
# Example usage
perplexity = calculate_perplexity(model, test_loader, device)
print("Perplexity:", perplexity)

In [ ]:
perplexity = calculate_perplexity(model, train_loader, device)
print("Perplexity:", perplexity)